# P09 — Token-aware Text Chunking

Notebook ini membaca JSON keluaran P08, membersihkan teks tanpa menghilangkan nomor halaman, memecah teks menjadi kalimat, lalu membentuk chunk sesuai tokenizer model summarization yang diputuskan pada P04.

## 1. Instalasi dan import

In [2]:
!pip -q install transformers sentencepiece

import json
import re
import unicodedata
from pathlib import Path
from transformers import AutoTokenizer
from google.colab import files

OUTPUT_DIR = Path("/content/data/processed_materials")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Konfigurasi tokenizer dan batas token

P04 menggunakan `cahya/t5-base-indonesian-summarization-cased`. Karena konfigurasi P04 mengizinkan input hingga 1024 token dan tidak menambahkan task prefix `ringkas:`, P09 memakai tokenizer dan batas tersebut agar konsisten. Margin keamanan disediakan untuk menghindari input tepat di batas maksimum.

In [3]:
MODEL_NAME = "cahya/t5-base-indonesian-summarization-cased"
ENCODER_MAX_TOKENS = 1024
SAFETY_MARGIN_TOKENS = 32
OVERLAP_TOKENS = 80

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_CHUNK_TOKENS = ENCODER_MAX_TOKENS - SAFETY_MARGIN_TOKENS

def count_tokens(text: str) -> int:
    return len(tokenizer.encode(text, add_special_tokens=True))

print(f"Model tokenizer : {MODEL_NAME}")
print(f"Batas encoder    : {ENCODER_MAX_TOKENS}")
print(f"Margin keamanan : {SAFETY_MARGIN_TOKENS}")
print(f"Anggaran chunk  : {MAX_CHUNK_TOKENS}")
print(f"Overlap target  : {OVERLAP_TOKENS}")


Model tokenizer : cahya/t5-base-indonesian-summarization-cased
Batas encoder    : 1024
Margin keamanan : 32
Anggaran chunk  : 992
Overlap target  : 80


## 3. Upload JSON hasil P08

In [4]:
print("Unggah file *_extracted_pages.json yang dihasilkan P08.")
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("Tidak ada file JSON yang diunggah.")
if len(uploaded) != 1:
    raise ValueError("Unggah satu file JSON hasil P08 saja.")

input_filename, raw_bytes = next(iter(uploaded.items()))
try:
    input_payload = json.loads(raw_bytes.decode("utf-8"))
except Exception as exc:
    raise ValueError(f"JSON tidak valid: {exc}") from exc

# Mendukung format JSON baru dari P08 dan format list lama.
if isinstance(input_payload, dict) and isinstance(input_payload.get("pages"), list):
    pages_list = input_payload["pages"]
elif isinstance(input_payload, list):
    pages_list = input_payload
else:
    raise ValueError("Struktur JSON tidak dikenali. Harus memiliki key 'pages' atau berupa list halaman.")

if not pages_list:
    raise ValueError("JSON tidak berisi halaman.")
for i, page in enumerate(pages_list):
    if "page" not in page or "text" not in page:
        raise ValueError(f"Item halaman ke-{i} harus memiliki field 'page' dan 'text'.")
    page["page"] = int(page["page"])
    page["text"] = str(page["text"] or "")

print(f"Berhasil membaca {len(pages_list)} halaman dari {input_filename}.")


Unggah file *_extracted_pages.json yang dihasilkan P08.


Saving Alternifa_-_Memahami_Informasi_Artikel_extracted_pages.json to Alternifa_-_Memahami_Informasi_Artikel_extracted_pages.json
Berhasil membaca 25 halaman dari Alternifa_-_Memahami_Informasi_Artikel_extracted_pages.json.


## 4. Pembersihan ringan dan pemisahan kalimat

Pembersihan mempertahankan batas paragraf sebagai pemisah. Pemisahan kalimat tidak mensyaratkan huruf kapital setelah tanda baca, dan beberapa singkatan umum serta angka desimal dilindungi agar tidak mudah terpotong.

In [5]:
ABBREVIATIONS = [
    "Dr.", "Prof.", "Ir.", "No.", "no.", "hlm.", "Hlm.", "dll.",
    "dsb.", "dkk.", "dst.", "S.Pd.", "M.Kom.", "S.Kom.", "a.n."
]

def clean_page_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text or "")
    text = text.replace("\x00", " ").replace("\r\n", "\n").replace("\r", "\n")
    # Sambungkan kata yang terpotong tanda hubung di akhir baris (mis. pem-\nbaca).
    text = re.sub(r"(?<=[A-Za-zÀ-ÿ])-\s*\n\s*(?=[A-Za-zÀ-ÿ])", "", text)
    # Hapus deretan titik/garis panjang dari daftar isi, tanpa menghapus tanda baca biasa.
    text = re.sub(r"[.·_—-]{4,}", " ", text)
    paragraphs = re.split(r"\n\s*\n+", text)
    cleaned = []
    for paragraph in paragraphs:
        paragraph = re.sub(r"[ \t]*\n[ \t]*", " ", paragraph)
        paragraph = re.sub(r"\s+", " ", paragraph).strip()
        if paragraph:
            cleaned.append(paragraph)
    return "\n".join(cleaned)

def split_into_sentences(text: str) -> list[str]:
    protected = text
    placeholders = {}
    for idx, abbr in enumerate(ABBREVIATIONS):
        key = f"ZZABBR{idx}ZZ"
        protected = protected.replace(abbr, abbr[:-1] + key)
        placeholders[key] = "."
    # Jangan memisahkan titik desimal. Pecah setelah .!? bila diikuti spasi.
    protected = re.sub(r"(?<=\d)\.(?=\d)", "ZZDECIMALZZ", protected)
    parts = re.split(r"(?<=[.!?])(?:[\"”’']?)[ \t]+", protected)
    result = []
    for part in parts:
        part = part.replace("ZZDECIMALZZ", ".")
        for key, value in placeholders.items():
            part = part.replace(key, value)
        part = part.strip(" \t\n")
        if part:
            result.append(part)
    return result


## 5. Bentuk rekaman kalimat dengan ID sumber

In [6]:
sentence_records = []
for page_item in pages_list:
    page_num = page_item["page"]
    cleaned = clean_page_text(page_item["text"])
    for paragraph_idx, paragraph in enumerate(cleaned.split("\n"), start=1):
        for sentence in split_into_sentences(paragraph):
            sentence_records.append({
                "sentence_id": f"S{len(sentence_records)+1:05d}",
                "page": page_num,
                "paragraph_id": f"P{page_num:04d}_{paragraph_idx:03d}",
                "text": sentence,
                "tokens": count_tokens(sentence)
            })

if not sentence_records:
    raise ValueError("Tidak ditemukan kalimat setelah pembersihan. Periksa hasil ekstraksi P08.")

print(f"Jumlah kalimat: {len(sentence_records)}")
print(f"Jumlah halaman dengan teks: {len({r['page'] for r in sentence_records})}")
print(f"Total token kalimat: {sum(r['tokens'] for r in sentence_records):,}")
print("Preview:")
for row in sentence_records[:5]:
    print(row["sentence_id"], f"(hal. {row['page']})", row["text"][:250])


Jumlah kalimat: 580
Jumlah halaman dengan teks: 25
Total token kalimat: 13,026
Preview:
S00001 (hal. 1) Pulang Mudik: Selebrasi Agama dan Budaya
S00002 (hal. 1) Acara pulang mudik Lebaran selalu disambut dengan antusias oleh masyarakat layaknya sebuah selebrasi kolosal.
S00003 (hal. 1) Pulang mudik bukan sekadar melepas rindu kampung halaman, melainkan juga dihayati sebagai rangkaian rasa syukur dengan berakhirnya ibadah puasa selama sebulan.
S00004 (hal. 1) Ada keyakinan bagi mereka yang berpuasa bahwa ampunan Tuhan yang dijanjikan dalam bulan Ramadhan akan terganjal jika antara anak dan orangtua dan antara sesama teman serta keluarga belum saling memaafkan.
S00005 (hal. 1) Makanya acara pulang mudik juga punya makna spiritual yang kental, di samping melepas rindu dan napak tilas dengan kampung halaman.


## 6. Pecah kalimat yang melebihi anggaran dan bentuk chunk

Kalimat yang sangat panjang dipecah menjadi fragmen token. Setiap fragmen tetap menyimpan `source_sentence_id` dan memiliki `fragment_id` tersendiri. Overlap ditambahkan hanya jika gabungan overlap + kalimat berikutnya masih muat dalam batas token.

In [7]:
def split_oversized_sentence(record: dict, max_tokens: int) -> list[dict]:
    token_ids = tokenizer.encode(record["text"], add_special_tokens=False)
    fragments = []
    for part_no, start in enumerate(range(0, len(token_ids), max_tokens - 8), start=1):
        ids = token_ids[start:start + (max_tokens - 8)]
        fragment_text = tokenizer.decode(ids, skip_special_tokens=True).strip()
        if fragment_text:
            fragments.append({
                **record,
                "fragment_id": f"{record['sentence_id']}_PART{part_no:02d}",
                "source_sentence_id": record["sentence_id"],
                "text": fragment_text,
                "tokens": count_tokens(fragment_text),
                "is_fragment": True
            })
    return fragments

normalized_records = []
for record in sentence_records:
    if record["tokens"] > MAX_CHUNK_TOKENS:
        normalized_records.extend(split_oversized_sentence(record, MAX_CHUNK_TOKENS))
    else:
        normalized_records.append({
            **record,
            "fragment_id": record["sentence_id"],
            "source_sentence_id": record["sentence_id"],
            "is_fragment": False
        })

def build_chunks(records: list[dict]) -> list[dict]:
    chunks = []
    current = []

    def text_of(items):
        return " ".join(item["text"] for item in items).strip()

    def finalize(items, chunk_no):
        text = text_of(items)
        ids = list(dict.fromkeys(item["source_sentence_id"] for item in items))
        pages = sorted({item["page"] for item in items})
        return {
            "chunk_id": f"CHUNK_{chunk_no:03d}",
            "token_count": count_tokens(text),
            "token_budget": MAX_CHUNK_TOKENS,
            "source_pages": pages,
            "sentence_ids": ids,
            "fragment_ids": [item["fragment_id"] for item in items],
            "text": text,
            "sentences_detail": [
                {"sentence_id": item["source_sentence_id"],
                 "fragment_id": item["fragment_id"],
                 "page": item["page"],
                 "paragraph_id": item["paragraph_id"],
                 "text": item["text"],
                 "is_fragment": item["is_fragment"]}
                for item in items
            ]
        }

    for record in records:
        candidate = current + [record]
        if count_tokens(text_of(candidate)) <= MAX_CHUNK_TOKENS:
            current = candidate
            continue

        if current:
            chunks.append(finalize(current, len(chunks) + 1))

        # Ambil ekor chunk sebelumnya sebagai overlap, maksimal target token.
        overlap = []
        for previous in reversed(current):
            trial = [previous] + overlap
            if count_tokens(text_of(trial)) <= OVERLAP_TOKENS:
                overlap = trial
            else:
                break
        current = overlap

        # Jika overlap membuat item baru tidak muat, mulai tanpa overlap.
        if count_tokens(text_of(current + [record])) > MAX_CHUNK_TOKENS:
            current = []
        current.append(record)

    if current:
        chunks.append(finalize(current, len(chunks) + 1))
    return chunks

chunks = build_chunks(normalized_records)
if not chunks:
    raise RuntimeError("Tidak ada chunk yang terbentuk.")

overflow = [c for c in chunks if c["token_count"] > MAX_CHUNK_TOKENS]
if overflow:
    raise RuntimeError(f"Ditemukan {len(overflow)} chunk melebihi anggaran token.")

print(f"Berhasil membentuk {len(chunks)} chunk; seluruhnya <= {MAX_CHUNK_TOKENS} token.")


Berhasil membentuk 14 chunk; seluruhnya <= 992 token.


## 7. Audit hasil chunking

In [8]:
for chunk in chunks:
    print(
        f"{chunk['chunk_id']} | {chunk['token_count']}/{MAX_CHUNK_TOKENS} token | "
        f"halaman={chunk['source_pages']} | "
        f"kalimat sumber={len(chunk['sentence_ids'])}"
    )

assert all(c["token_count"] <= MAX_CHUNK_TOKENS for c in chunks)
assert all(c["text"].strip() for c in chunks)
print("AUDIT LULUS: tidak ada chunk kosong atau melebihi batas token.")


CHUNK_001 | 989/992 token | halaman=[1, 2, 3] | kalimat sumber=50
CHUNK_002 | 990/992 token | halaman=[2, 3, 4] | kalimat sumber=43
CHUNK_003 | 983/992 token | halaman=[4, 5, 6] | kalimat sumber=42
CHUNK_004 | 983/992 token | halaman=[6, 7, 8] | kalimat sumber=53
CHUNK_005 | 972/992 token | halaman=[8, 9, 10] | kalimat sumber=49
CHUNK_006 | 978/992 token | halaman=[10, 11, 12] | kalimat sumber=45
CHUNK_007 | 989/992 token | halaman=[12, 13, 14] | kalimat sumber=45
CHUNK_008 | 977/992 token | halaman=[14, 15, 16] | kalimat sumber=51
CHUNK_009 | 991/992 token | halaman=[16, 17, 18] | kalimat sumber=58
CHUNK_010 | 990/992 token | halaman=[18, 19, 20] | kalimat sumber=49
CHUNK_011 | 968/992 token | halaman=[20, 21] | kalimat sumber=40
CHUNK_012 | 981/992 token | halaman=[21, 22] | kalimat sumber=32
CHUNK_013 | 962/992 token | halaman=[22, 23, 24] | kalimat sumber=39
CHUNK_014 | 575/992 token | halaman=[24, 25] | kalimat sumber=28
AUDIT LULUS: tidak ada chunk kosong atau melebihi batas toke

## 8. Simpan dan unduh output

In [9]:
source_stem = Path(input_filename).stem.replace("_extracted_pages", "")
output_file = OUTPUT_DIR / f"{source_stem}_token_chunks.json"
output_payload = {
    "pipeline_metadata": {
        "source_file": input_filename,
        "tokenizer_model": MODEL_NAME,
        "encoder_max_tokens": ENCODER_MAX_TOKENS,
        "safety_margin_tokens": SAFETY_MARGIN_TOKENS,
        "max_chunk_tokens": MAX_CHUNK_TOKENS,
        "overlap_target_tokens": OVERLAP_TOKENS,
        "total_pages": len(pages_list),
        "total_sentences": len(sentence_records),
        "total_chunks": len(chunks)
    },
    "chunks": chunks
}
with output_file.open("w", encoding="utf-8") as f:
    json.dump(output_payload, f, ensure_ascii=False, indent=2)

print(f"Output tersimpan: {output_file}")
files.download(str(output_file))


Output tersimpan: /content/data/processed_materials/Alternifa_-_Memahami_Informasi_Artikel_token_chunks.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Catatan penggunaan

- Input notebook ini adalah JSON keluaran P08, bukan PDF langsung.
- Token dihitung dengan tokenizer model summarization P04 dan special token diperhitungkan.
- `sentence_ids` dan `fragment_ids` mempertahankan jejak ke sumber.
- Overlap dapat membuat satu kalimat sumber muncul di dua chunk; ini disengaja untuk menjaga konteks.
- Untuk integrasi berikutnya, simpan JSON chunk ke `data/processed_materials/` pada repository.